# 29 — Multi-Scale Sync Dynamics Under FIM

Does FIM create hierarchical synchronisation (local domains first, then global)
or does it enforce direct global sync?

This matters because:
- Hierarchical: consistent with Miroslav's observation scales theory
- Direct global: suggests FIM acts as a mean-field override

Tests:
1. **Time-resolved R per domain** — track R for sub-groups of oscillators
2. **Sync onset ordering** — which pairs synchronise first?
3. **Cluster formation** — does the system fragment into meta-stable clusters before full sync?
4. **Frequency entrainment** — do effective frequencies converge uniformly or hierarchically?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

# Natural domain structure from K_nm:
# Strongly coupled groups (based on K_nm > 0.2):
# Domain A: L1-L4 (autonomic, low frequency)
# Domain B: L5-L8 (intermediate)
# Domain C: L9-L12 (cortical low)
# Domain D: L13-L16 (cortical high + strange loop)
DOMAINS = {
    "A (L1-L4)": list(range(0, 4)),
    "B (L5-L8)": list(range(4, 8)),
    "C (L9-L12)": list(range(8, 12)),
    "D (L13-L16)": list(range(12, 16)),
}


def fim_gradient(phases, i):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases[i] + np.pi) % (2 * np.pi) - np.pi
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    return (1.0 / n) * np.sin(phase_diff) * min(sensitivity, 50.0)


def simulate_trajectory_full(
    K_scale, fim_lambda, dt=0.02, T=200.0, noise=0.05, seed=42, sample_every=25
):
    """Full trajectory with sampling for time-resolved analysis."""
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)

    times = []
    R_global = []
    R_domains = {name: [] for name in DOMAINS}
    freq_eff = []  # effective frequencies
    theta_prev = theta.copy()

    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if fim_lambda > 0:
            for i in range(N):
                dphi[i] += fim_lambda * fim_gradient(theta, i)
        theta_new = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)

        if s % sample_every == 0:
            t = s * dt
            times.append(t)

            # Global R
            z = np.exp(1j * theta_new)
            R_global.append(float(np.abs(np.mean(z))))

            # Domain R
            for name, indices in DOMAINS.items():
                z_dom = np.exp(1j * theta_new[indices])
                R_domains[name].append(float(np.abs(np.mean(z_dom))))

            # Effective frequency
            dtheta = theta_new - theta_prev
            # Unwrap
            dtheta = (dtheta + np.pi) % (2 * np.pi) - np.pi
            freq_eff.append(dtheta / (sample_every * dt))

        theta_prev = theta_new.copy()
        theta = theta_new

    return {
        "times": np.array(times),
        "R_global": np.array(R_global),
        "R_domains": {k: np.array(v) for k, v in R_domains.items()},
        "freq_eff": np.array(freq_eff),
    }


print(f"Domains: {list(DOMAINS.keys())}")
print("Ready.")

In [ ]:
# Run for 3 configurations: no FIM, FIM λ=1, FIM λ=5
K_SCALE = 12  # sub-threshold for no-FIM sync

configs = [
    ("no FIM", 0.0),
    ("FIM λ=1", 1.0),
    ("FIM λ=5", 5.0),
]

all_results = {}
for name, lam in configs:
    print(f"\n=== {name} (K_scale={K_SCALE}) ===")
    traj = simulate_trajectory_full(K_SCALE, lam, T=200.0)
    all_results[name] = traj

    # When does each domain reach R > 0.7?
    print(f"  Global R final: {traj['R_global'][-1]:.4f}")
    for dom_name, R_dom in traj["R_domains"].items():
        # Find first time R > 0.7
        idx = np.where(np.array(R_dom) > 0.7)[0]
        if len(idx) > 0:
            t_sync = traj["times"][idx[0]]
            print(f"  {dom_name}: R>0.7 at t={t_sync:.1f}, R_final={R_dom[-1]:.4f}")
        else:
            print(f"  {dom_name}: NEVER reaches R>0.7, R_final={R_dom[-1]:.4f}")

In [ ]:
# Sync onset ordering analysis
print("\n=== SYNC ONSET ORDERING ===")
print("Does FIM change the order in which domains synchronise?\n")

for name, _lam in configs:
    traj = all_results[name]
    onset_times = {}
    for dom_name, R_dom in traj["R_domains"].items():
        idx = np.where(np.array(R_dom) > 0.6)[0]
        onset_times[dom_name] = float(traj["times"][idx[0]]) if len(idx) > 0 else float("inf")

    # Sort by onset time
    order = sorted(onset_times.items(), key=lambda x: x[1])
    print(f"{name}:")
    for dom, t in order:
        print(f"  {t:6.1f}s — {dom}")
    print()

In [ ]:
# Frequency entrainment analysis
print("=== FREQUENCY ENTRAINMENT ===")
print("Do effective frequencies converge to a common value?\n")

for name, _lam in configs:
    traj = all_results[name]
    freq = traj["freq_eff"]

    # Early (first 20%) vs late (last 20%) frequency spread
    n_t = len(freq)
    early = freq[: n_t // 5]
    late = freq[4 * n_t // 5 :]

    spread_early = np.std(np.mean(early, axis=0))
    spread_late = np.std(np.mean(late, axis=0))

    # Per-domain frequency spread
    print(f"{name}:")
    print(
        f"  Frequency spread: early={spread_early:.4f}, late={spread_late:.4f}, reduction={1 - spread_late / (spread_early + 1e-10):.1%}"
    )
    for dom_name, indices in DOMAINS.items():
        dom_spread_late = np.std(np.mean(late[:, indices], axis=0))
        print(f"  {dom_name}: intra-domain spread = {dom_spread_late:.4f}")
    print()

In [ ]:
# Pairwise PLV over time — which oscillator pairs lock first?
print("=== PAIRWISE PHASE LOCKING (FIM λ=5) ===")
traj = all_results["FIM λ=5"]
times = traj["times"]

# Compute PLV at early (t=10), mid (t=50), late (t=150)
for t_target in [10, 50, 150]:
    idx = np.argmin(np.abs(times - t_target))
    # We need a window of phases around this time
    window = 20
    start = max(0, idx - window)
    end = min(len(times), idx + window)

    # Simple: use the snapshot phase differences
    # Re-run short trajectory to get instantaneous phases

# Simpler approach: measure global vs domain R convergence profile
print("\nConvergence profile (FIM λ=5, K=12):")
print(f"{'t':>6} {'R_global':>8}", end="")
for dom in DOMAINS:
    print(f" {dom[:5]:>8}", end="")
print()

traj = all_results["FIM λ=5"]
for i in range(0, len(traj["times"]), max(1, len(traj["times"]) // 15)):
    print(f"{traj['times'][i]:6.1f} {traj['R_global'][i]:8.4f}", end="")
    for dom in DOMAINS:
        print(f" {traj['R_domains'][dom][i]:8.4f}", end="")
    print()

# Key question: do domains sync before global?
print("\n=== VERDICT ===")
traj5 = all_results["FIM λ=5"]
dom_R_at_half = {}
t_half_idx = len(traj5["times"]) // 2
for dom in DOMAINS:
    dom_R_at_half[dom] = traj5["R_domains"][dom][t_half_idx]
global_R_at_half = traj5["R_global"][t_half_idx]

domains_ahead = sum(1 for v in dom_R_at_half.values() if v > global_R_at_half + 0.05)
if domains_ahead >= 3:
    print("HIERARCHICAL: Domains synchronise BEFORE global sync.")
    print("FIM orchestrates bottom-up assembly.")
elif global_R_at_half > max(dom_R_at_half.values()) - 0.05:
    print("DIRECT GLOBAL: FIM enforces uniform global sync.")
    print("No intermediate domain structure.")
else:
    print("MIXED: Some domains lead, others follow.")

In [ ]:
# Save
output = {
    "experiment": "Multi-scale sync dynamics under FIM",
    "N": N,
    "K_scale": K_SCALE,
    "domains": {k: v for k, v in DOMAINS.items()},
}
for name, traj in all_results.items():
    output[name] = {
        "R_global_final": round(float(traj["R_global"][-1]), 4),
        "R_domains_final": {k: round(float(v[-1]), 4) for k, v in traj["R_domains"].items()},
    }

out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/multiscale_sync_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")